# Chapter 5: Cost, Performance, and Model SelectionEstimated time: ~8 hours (including a ~2-2.5 hour conceptual, no-training fine-tuningsubsection near the end).Prerequisites: Chapter 1 (`agentlib.llm_client`).

## SetupThis chapter uses real tokenization via `tiktoken`. `tiktoken.get_encoding("cl100k_base")`normally downloads its vocabulary file from `openaipublic.blob.core.windows.net` on firstuse. This repo vendors a hash-verified copy in `data/tiktoken_cache/`, pre-seeded into`tiktoken`'s own local cache directory. See `PROGRESS.md` for verification details.

In [1]:
import sys
from pathlib import Path

_repo_root = Path.cwd()
if not (_repo_root / "agentlib").is_dir():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

import os
os.environ["TIKTOKEN_CACHE_DIR"] = str(_repo_root / "data" / "tiktoken_cache")

import random

import tiktoken

from agentlib.grading import check
from agentlib import llm_client, synthetic_data

random.seed(42)
encoding = tiktoken.get_encoding("cl100k_base")
print("tiktoken cl100k_base loaded from local cache -- no network call made.")
print(f"LLM_PROVIDER = {llm_client.LLM_PROVIDER!r}, HAS_KEY = {llm_client.HAS_KEY}")
print("This chapter makes no model calls: the cost and latency work below runs against a\n"
      "synthetic request log and real tokenizer output, so the numbers are reproducible\n"
      "and cost nothing to re-derive. HAS_KEY is printed for consistency with the other\n"
      "chapters, not because anything here branches on it.")

tiktoken cl100k_base loaded from local cache -- no network call made.
LLM_PROVIDER = 'anthropic', HAS_KEY = False
This chapter makes no model calls: the cost and latency work below runs against a
synthetic request log and real tokenizer output, so the numbers are reproducible
and cost nothing to re-derive. HAS_KEY is printed for consistency with the other
chapters, not because anything here branches on it.


## Section 1: Definitions### Token economicsInput and output tokens are priced separately. Output is almost always more expensive pertoken: generation is autoregressive (each token depends on computing every token beforeit), while input tokens are processed in parallel during prefill. A long system promptsent on every call is a fixed tax on every request; a verbose model generating longanswers pays that tax on the expensive side.Real-world pricing (illustrative, mid-2025):| Model tier | Input (per 1M tokens) | Output (per 1M tokens) | Ratio ||---|---|---|---|| Haiku-class (fast/cheap) | ~$0.25-1.00 | ~$1.25-5.00 | 5x || Sonnet-class (balanced) | ~$3.00 | ~$15.00 | 5x || Opus-class (strongest) | ~$15.00 | ~$75.00 | 5x |### Prompt cachingIf a large chunk of the prompt repeats across calls (a long system prompt, a retrieveddocument reused across turns), paying full prefill cost every time is waste. A cache hitskips re-processing the cached prefix; only the new suffix costs full price.Real-world examples:- **Anthropic prompt caching**: cached input tokens billed at ~10% of full input price.  A 2,000-token system prompt reused across 10,000 calls saves ~$54 at Sonnet pricing.- **OpenAI batch API**: 50% discount on input tokens for non-real-time workloads.- **System prompt reuse**: any application where the same instructions prefix every call.### Model routingRoute each request to the cheapest model that can still do the task correctly. Do not paysenior-employee rates for a task any junior employee could handle, and do not hand asenior-level task to a junior employee just because they are cheaper per hour.Real-world examples:- **Classification then generation**: use Haiku to classify intent, route complex queries  to Sonnet, keep simple queries on Haiku. One provider, two tiers.- **Tool-use escalation**: simple Q&A stays on the cheap tier; multi-step tool loops  escalate to the strong tier where mistakes compound.- **Length-based routing**: short prompts (likely simple tasks) stay cheap; long prompts  (likely complex analysis) escalate.### Fine-tuning concepts (LoRA, QLoRA, RLHF)| Term | What it is | When to reach for it ||---|---|---|| Full fine-tuning | Update every weight in the model | Rarely; expensive, produces a full-size copy per task || LoRA (Hu et al., 2021) | Freeze base weights, train small low-rank matrices alongside them | Stable repeated behavior that prompting cannot produce reliably || QLoRA (Dettmers et al., 2023) | LoRA + quantize frozen base to 4-bit during training | Same as LoRA but on a single consumer GPU || RLHF (Christiano et al., 2017) | Shape model behavior with human preference judgments | How the base model you call was trained; not something you typically do yourself |Reach for fine-tuning only after ruling out better prompting, better retrieval, or bettertool design. Prompting ships in minutes and is reversible; a fine-tune is a multi-hourtraining run that produces a new artifact to version and serve.

## Section 2: Concept Explanation### Four-stage latency decomposition"The p99 got worse" is not a diagnosis. Four separable things happen between a requestarriving and a token reaching the user:```  Request arrives       |       v  +-----------+     +-----------+     +------------+     +-------------+  |  QUEUE    | --> |  NETWORK  | --> |  INFERENCE  | --> | GENERATION  |  | (waiting  |     | (on the   |     | (processing |     | (producing  |  |  for GPU) |     |  wire)    |     |  the prompt)|     |  output     |  +-----------+     +-----------+     +------------+     |  tokens)    |                                                          +-------------+                                                               |                                                               v                                                         Token reaches user```Each stage has different causes and different fixes:| Stage | What actually helps ||---|---|| Queueing | More capacity, backpressure, load shedding || Network | Regional routing, connection reuse || Inference (prefill) | Prompt caching, shorter prompts || Generation (decode) | Shorter outputs, a faster/smaller model |A regression lands in exactly one of them. Diagnosing which one requires looking at stageshares (what percentage of total latency each stage accounts for), not just totals.### Tokenization quirksReal prompts do not tokenize the way you would guess by counting words:- Punctuation usually splits into its own token- Numbers split at arbitrary boundaries learned from training data- A single long uncommon word can cost more tokens than an entire short sentence- Repeated whitespace collapses into a single token (cl100k_base has dedicated merges)- Code indentation is cheap; GPT-2's older tokenizer burned one token per space### Trade-offs| Decision | Cheap/fast option | Expensive/thorough option ||---|---|---|| Prompt caching | Setup cost, cache invalidation complexity | Pays for itself after a few hundred calls || Model routing | Misrouted complex tasks fail expensively | Router adds a decision point to maintain || Context window | Truncate history (lose recall) | Send everything (linear cost growth) || Fine-tuning | Multi-hour training, versioning overhead | Ship prompt changes in minutes instead |

## Section 3: Example Code SegmentsSynthetic request log generation, real tokenization, and the injection functions used bythe break-it scenarios.

### Synthetic request log`agentlib.synthetic_data.generate_request_log()` produces a realistic-shaped log: Poissonarrivals, log-normal token counts, and a four-stage latency breakdown per request. Seededfor reproducibility.

In [2]:
log = synthetic_data.generate_request_log(n_requests=500, seed=42)

print(f"{len(log)} requests logged.")
print("First request: ", log[0])
avg_latency = sum(r["total_latency_ms"] for r in log) / len(log)
print(f"\nAverage total latency: {avg_latency:.0f}ms -- this is this chapter's 'normal' baseline.")


500 requests logged.
First request:  {'request_id': 'req-00000', 'timestamp': 2.04, 'model': 'haiku', 'input_tokens': 299, 'output_tokens': 227, 'queue_depth': 2, 'queue_time_ms': 73.8, 'network_time_ms': 10.9, 'inference_time_ms': 39.7, 'generation_time_ms': 2619.0, 'total_latency_ms': 2743.4}

Average total latency: 1758ms -- this is this chapter's 'normal' baseline.


### Real tokenization with tiktokenReal prompts on a handful of deliberately quirky examples. Watch how punctuation, numbers,whitespace, and uncommon words tokenize differently than you would guess.

In [3]:
examples = [
    "hello world",
    "Hello, world!",
    "The invoice total is $1,234.56.",
    "def calculate_total(items, discount_code=None):",
    "                                        ",  # lots of whitespace
    "supercalifragilisticexpialidocious",
]

for text in examples:
    tokens = encoding.encode(text)
    pieces = [encoding.decode([t]) for t in tokens]
    print(f"{text!r}")
    print(f"  {len(tokens)} tokens: {pieces}")
    print()


'hello world'
  2 tokens: ['hello', ' world']

'Hello, world!'
  4 tokens: ['Hello', ',', ' world', '!']

'The invoice total is $1,234.56.'
  11 tokens: ['The', ' invoice', ' total', ' is', ' $', '1', ',', '234', '.', '56', '.']

'def calculate_total(items, discount_code=None):'
  9 tokens: ['def', ' calculate', '_total', '(items', ',', ' discount', '_code', '=None', '):']

'                                        '
  1 tokens: ['                                        ']

'supercalifragilisticexpialidocious'
  11 tokens: ['sup', 'erc', 'al', 'if', 'rag', 'il', 'istic', 'exp', 'ial', 'id', 'ocious']



Punctuation attached to a word usually splits into its own token. Numbers split atarbitrary frequency-based boundaries. A single long uncommon word can cost 11 tokens (thesame as a 6-word sentence). A long run of whitespace collapses into one token. `cl100k_base`has dedicated merges for repeated whitespace, which matters for indented code.

### Bug injection functionsThree functions that each inject a different pathology into a slice of the request log.Used by the break-it section; shown here so you can read them before diagnosing.

In [6]:
def inject_duplicate_calls(log: list, start: int, count: int) -> list:
    '''The bug: a retry path that isn't idempotent double-executes the call, so both
    input and output tokens are ~2x what a normal request of this kind costs -- but it's
    still logged as a single request.'''
    injected = [dict(r) for r in log]
    for i in range(start, start + count):
        injected[i] = dict(injected[i])
        injected[i]["input_tokens"] *= 2
        injected[i]["output_tokens"] *= 2
        injected[i]["inference_time_ms"] = round(injected[i]["input_tokens"] * 0.175, 1)
        injected[i]["generation_time_ms"] = round(injected[i]["output_tokens"] * 11.5, 1)
        injected[i]["total_latency_ms"] = round(
            injected[i]["queue_time_ms"] + injected[i]["network_time_ms"]
            + injected[i]["inference_time_ms"] + injected[i]["generation_time_ms"], 1
        )
    return injected


scenario_1_log = inject_duplicate_calls(log, start=100, count=20)

for r in scenario_1_log[95:105]:
    print(f"{r['request_id']}  input_tokens={r['input_tokens']:5d}  output_tokens={r['output_tokens']:4d}")


req-00095  input_tokens=  285  output_tokens=  88
req-00096  input_tokens=  484  output_tokens= 165
req-00097  input_tokens=  644  output_tokens= 142
req-00098  input_tokens=  179  output_tokens= 207
req-00099  input_tokens=  490  output_tokens= 192
req-00100  input_tokens= 2074  output_tokens= 276
req-00101  input_tokens= 2086  output_tokens= 244
req-00102  input_tokens=  568  output_tokens= 150
req-00103  input_tokens=  552  output_tokens= 172
req-00104  input_tokens= 1576  output_tokens= 172


In [9]:
def inject_unbounded_context_growth(log: list, start: int, count: int, growth_per_turn: int = 180) -> list:
    '''The bug: a conversational agent that appends every prior turn to the prompt with no
    truncation -- input_tokens grows roughly linearly across a session instead of staying
    in its normal range.'''
    injected = [dict(r) for r in log]
    for offset in range(count):
        i = start + offset
        injected[i] = dict(injected[i])
        injected[i]["input_tokens"] = 150 + offset * growth_per_turn
        injected[i]["inference_time_ms"] = round(injected[i]["input_tokens"] * 0.175, 1)
        injected[i]["total_latency_ms"] = round(
            injected[i]["queue_time_ms"] + injected[i]["network_time_ms"]
            + injected[i]["inference_time_ms"] + injected[i]["generation_time_ms"], 1
        )
    return injected


scenario_2_log = inject_unbounded_context_growth(log, start=200, count=30)

for r in scenario_2_log[200:230:5]:
    print(f"{r['request_id']}  input_tokens={r['input_tokens']:5d}  inference_time_ms={r['inference_time_ms']:7.1f}")


req-00200  input_tokens=  150  inference_time_ms=   26.2
req-00205  input_tokens= 1050  inference_time_ms=  183.8
req-00210  input_tokens= 1950  inference_time_ms=  341.2
req-00215  input_tokens= 2850  inference_time_ms=  498.7
req-00220  input_tokens= 3750  inference_time_ms=  656.2
req-00225  input_tokens= 4650  inference_time_ms=  813.8


In [12]:
def inject_queueing_spike(log: list, start: int, count: int, peak_queue_depth: int = 42) -> list:
    '''The bug: a burst of concurrent traffic (a batch job, a retry storm, a traffic spike)
    overwhelms available capacity. Nothing about any individual request is unusual -- its
    own input/output tokens are normal -- but queue_depth spikes, so queue_time_ms dominates
    total latency.'''
    injected = [dict(r) for r in log]
    for offset in range(count):
        i = start + offset
        injected[i] = dict(injected[i])
        # Ramp queue depth up and back down across the window (a bursty spike, not a step).
        progress = offset / (count - 1)
        depth = int(peak_queue_depth * (1 - abs(2 * progress - 1)))
        injected[i]["queue_depth"] = depth
        injected[i]["queue_time_ms"] = round(depth * 270.0, 1)
        injected[i]["total_latency_ms"] = round(
            injected[i]["queue_time_ms"] + injected[i]["network_time_ms"]
            + injected[i]["inference_time_ms"] + injected[i]["generation_time_ms"], 1
        )
    return injected


scenario_3_log = inject_queueing_spike(log, start=300, count=30)

baseline_latency = sum(r["total_latency_ms"] for r in log[270:300]) / 30
spike_latency = max(r["total_latency_ms"] for r in scenario_3_log[300:330])
print(f"Baseline total_latency_ms (pre-spike window): {baseline_latency:.0f}ms")
print(f"Peak total_latency_ms (inside the spike):      {spike_latency:.0f}ms")
print()
for r in scenario_3_log[300:330:4]:
    print(
        f"{r['request_id']}  queue_depth={r['queue_depth']:3d}  "
        f"queue_time_ms={r['queue_time_ms']:7.1f}  input_tokens={r['input_tokens']:5d}  "
        f"total_latency_ms={r['total_latency_ms']:8.1f}"
    )


Baseline total_latency_ms (pre-spike window): 1696ms
Peak total_latency_ms (inside the spike):      12428ms

req-00300  queue_depth=  0  queue_time_ms=    0.0  input_tokens=  544  total_latency_ms=  1231.3
req-00304  queue_depth= 11  queue_time_ms= 2970.0  input_tokens=  306  total_latency_ms=  5159.5
req-00308  queue_depth= 23  queue_time_ms= 6210.0  input_tokens=  430  total_latency_ms=  8171.7
req-00312  queue_depth= 34  queue_time_ms= 9180.0  input_tokens=  392  total_latency_ms= 10078.5
req-00316  queue_depth= 37  queue_time_ms= 9990.0  input_tokens=  388  total_latency_ms= 11182.0
req-00320  queue_depth= 26  queue_time_ms= 7020.0  input_tokens=  726  total_latency_ms=  8016.8
req-00324  queue_depth= 14  queue_time_ms= 3780.0  input_tokens=  195  total_latency_ms=  5619.3
req-00328  queue_depth=  2  queue_time_ms=  540.0  input_tokens=  412  total_latency_ms=  2136.6


## Section 4: Build It YourselfSix graded tasks: a latency profiler, a cost estimator with caching, a model router, andthree written diagnoses of production pathologies.

### Task 1: `profile_latency` (four-stage decomposition)Decompose a request log into four stages as totals AND as shares of the whole. The shareis the diagnostic half: a stage that grew from 10% to 60% names the culprit.

In [ ]:
def profile_latency(requests: list) -> dict:
    '''Decompose a request log into the four stages, as totals AND as shares of the whole.

    Each request carries queue_time_ms, network_time_ms, inference_time_ms and
    generation_time_ms. Return:

        {stage: {"total_ms": <summed across every request>,
                 "pct": <that total as a percentage of all four stages combined>}}

    pct is a share of the grand total, not a mean -- the four shares add up to 100.

    An empty log, or one where every stage is zero, reports zeros rather than dividing by
    zero: a quiet window is a normal thing for a profiler to be handed.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


profile_latency = check("ch05-latency-profile", profile_latency)

#### Profiler output on the baseline log

In [5]:
profile = profile_latency(log)
print(f"{'stage':20s} {'total_ms':>12s} {'% of total':>12s}")
for stage, stats in profile.items():
    print(f"{stage:20s} {stats['total_ms']:>12.0f} {stats['pct']:>11.1f}%")

stage                    total_ms   % of total
queue_time_ms               41526         4.7%
network_time_ms             12286         1.4%
inference_time_ms           42911         4.9%
generation_time_ms         782329        89.0%


### Task 2: `estimate_cost_with_caching` (prompt caching cost model)Two things this has to get right: input and output tokens are priced differently (outputis 5x input), and cached tokens are cheap, not free.

In [ ]:
def estimate_cost_with_caching(input_tokens: int, output_tokens: int, cached_prefix_tokens: int,
                                cache_hit: bool, input_price=3.0, output_price=15.0, cache_price=0.30) -> dict:
    '''Per-million-token prices in dollars (illustrative, Sonnet-class ballpark).

    On a cache MISS you pay full input price for every input token, prefix included, and the
    cached_prefix_tokens figure is irrelevant.

    On a cache HIT the prefix is billed at cache_price instead of input_price, and only the
    remaining (input_tokens - cached_prefix_tokens) count as fresh input.

    Output is always billed at output_price -- note it is 5x the input price here, which is
    why pricing both sides the same understates exactly the verbose responses you most want
    a cost model to flag.

    Return {"cache_hit": ..., "fresh_input_tokens": ..., "cost_usd": round(cost, 6)}.
    Prices are per MILLION tokens.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


estimate_cost_with_caching = check("ch05-token-cost", estimate_cost_with_caching)

#### Cost with and without caching

In [17]:
# A long, reused system prompt (2,000 tokens) plus a short per-call question (50 tokens).
no_cache = estimate_cost_with_caching(input_tokens=2050, output_tokens=300, cached_prefix_tokens=2000, cache_hit=False)
with_cache = estimate_cost_with_caching(input_tokens=2050, output_tokens=300, cached_prefix_tokens=2000, cache_hit=True)

print("Without caching:", no_cache)
print("With caching:   ", with_cache)
savings_per_call = no_cache["cost_usd"] - with_cache["cost_usd"]
print(f"\nSavings per call: ${savings_per_call:.6f}  ->  over 10,000 calls: ${savings_per_call * 10000:.2f}")

Without caching: {'cache_hit': False, 'fresh_input_tokens': 2050, 'cost_usd': 0.01065}
With caching:    {'cache_hit': True, 'fresh_input_tokens': 50, 'cost_usd': 0.00525}

Savings per call: $0.005400  ->  over 10,000 calls: $54.00


### Task 3: `route_request` (model routing)Route to the strong tier if the request needs multi-step tool use OR is longer than 80words. Two independent escalation reasons; the second (tool use) is the one that actuallycosts money when you get it wrong.

In [ ]:
def route_request(prompt: str, requires_tool_use: bool = False) -> str:
    '''A simple, legible routing policy.

    Send a request to the strong tier if EITHER it needs multi-step tool use OR it is longer
    than 80 words (counted in words, not characters). Otherwise the cheap/fast tier.
    Exactly 80 words is not yet over the line.

    Return an actual model id -- llm_client.STRONG_MODELS[llm_client.LLM_PROVIDER] or
    llm_client.DEFAULT_MODELS[llm_client.LLM_PROVIDER] -- not a tier label, since the caller
    passes this straight to call_model().

    A real production router would likely use a small classifier or a cheap model call to
    make this decision; the policy itself is the point here, not the classifier.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


route_request = check("ch05-router", route_request)

#### Routing in action

In [19]:
test_requests = [
    ("What's the capital of France?", False),
    ("Summarize this 40-page contract, flag every clause that deviates from our standard "
     "template, and cross-reference each flagged clause against the three most similar "
     "clauses from our last 20 signed contracts.", True),
    ("Format this as a bulleted list.", False),
]

for prompt, needs_tools in test_requests:
    model = route_request(prompt, needs_tools)
    print(f"[{model}]  {prompt[:60]}{'...' if len(prompt) > 60 else ''}")

[claude-haiku-4-5-20251001]  What's the capital of France?
[claude-sonnet-5]  Summarize this 40-page contract, flag every clause that devi...
[claude-haiku-4-5-20251001]  Format this as a bulleted list.


## Section 5: PlaygroundExperiments with editable parameters. Change the values marked `EDIT THESE`, re-run thecell, and observe how the behavior changes.

### Experiment 1: Cache hit rate and cost savingsHow much does caching save at different hit rates? Try changing the system prompt lengthand per-call question length.

In [ ]:
# --- EDIT THESE ---SYSTEM_PROMPT_TOKENS = 2000  # try [500, 1000, 4000, 8000]QUESTION_TOKENS = 50  # try [20, 100, 200]OUTPUT_TOKENS = 300  # try [100, 500, 1000]HIT_RATES = [0.0, 0.5, 0.9, 1.0]  # fraction of calls that hit cacheprint(f"System prompt: {SYSTEM_PROMPT_TOKENS} tokens, question: {QUESTION_TOKENS} tokens")print(f"Output: {OUTPUT_TOKENS} tokens per call")print()print(f"{'Hit rate':>10}  {'Cost/call':>12}  {'Cost/10K calls':>15}  {'Savings vs 0%':>14}")print("-" * 58)input_tokens = SYSTEM_PROMPT_TOKENS + QUESTION_TOKENSbaseline = estimate_cost_with_caching(input_tokens, OUTPUT_TOKENS, SYSTEM_PROMPT_TOKENS, cache_hit=False)for rate in HIT_RATES:    miss_cost = baseline["cost_usd"]    hit_cost = estimate_cost_with_caching(input_tokens, OUTPUT_TOKENS, SYSTEM_PROMPT_TOKENS, cache_hit=True)["cost_usd"]    avg_cost = rate * hit_cost + (1 - rate) * miss_cost    savings = (1 - avg_cost / miss_cost) * 100    print(f"{rate:>10.0%}  ${avg_cost:>11.6f}  ${avg_cost * 10000:>14.2f}  {savings:>13.1f}%")

### Experiment 2: Model tier cost comparisonHow much does routing save compared to sending everything to the strong tier?

In [ ]:
# --- EDIT THESE ---TOTAL_REQUESTS = 1000SIMPLE_FRACTION = 0.7  # try [0.3, 0.5, 0.8, 0.95]AVG_INPUT_TOKENS = 500AVG_OUTPUT_TOKENS = 200# Illustrative per-million-token pricesHAIKU_INPUT, HAIKU_OUTPUT = 1.0, 5.0SONNET_INPUT, SONNET_OUTPUT = 3.0, 15.0simple = int(TOTAL_REQUESTS * SIMPLE_FRACTION)complex_ = TOTAL_REQUESTS - simpleall_sonnet = TOTAL_REQUESTS * (AVG_INPUT_TOKENS * SONNET_INPUT + AVG_OUTPUT_TOKENS * SONNET_OUTPUT) / 1e6routed = (simple * (AVG_INPUT_TOKENS * HAIKU_INPUT + AVG_OUTPUT_TOKENS * HAIKU_OUTPUT)          + complex_ * (AVG_INPUT_TOKENS * SONNET_INPUT + AVG_OUTPUT_TOKENS * SONNET_OUTPUT)) / 1e6print(f"{TOTAL_REQUESTS} requests: {simple} simple ({SIMPLE_FRACTION:.0%}), {complex_} complex")print(f"All-Sonnet cost:  ${all_sonnet:.4f}")print(f"Routed cost:      ${routed:.4f}")print(f"Savings:          ${all_sonnet - routed:.4f} ({(1 - routed/all_sonnet)*100:.1f}%)")

### Experiment 3: Context window size and cost scalingHow does conversation length affect per-call cost when history is not truncated?

In [ ]:
# --- EDIT THESE ---TURNS = [1, 5, 10, 20, 50]  # try [2, 8, 15, 30]TOKENS_PER_TURN = 180  # try [100, 250, 500]SYSTEM_TOKENS = 500OUTPUT_TOKENS = 200print(f"{'Turns':>6}  {'Input tokens':>13}  {'Cost/call':>12}  {'vs turn 1':>10}")print("-" * 48)base_cost = Nonefor t in TURNS:    input_t = SYSTEM_TOKENS + t * TOKENS_PER_TURN    cost = estimate_cost_with_caching(input_t, OUTPUT_TOKENS, 0, cache_hit=False)["cost_usd"]    if base_cost is None:        base_cost = cost    ratio = cost / base_cost    print(f"{t:>6}  {input_t:>13}  ${cost:>11.6f}  {ratio:>9.1f}x")print()print("Without truncation, cost grows linearly with conversation length.")print("A 50-turn conversation costs ~7x more per call than a 1-turn conversation.")

### Experiment 4: Tokenization cost of different content typesHow many tokens do different kinds of content cost? Try your own examples.

In [ ]:
# --- EDIT THESE ---TEST_STRINGS = [    "What is the capital of France?",    "Please summarize the following document and extract all key findings.",    "def calculate_total(items, discount_code=None):\n    total = sum(i['price'] for i in items)\n    return total",    "The quarterly revenue was $4,521,890.23 for Q3 2025, representing a 12.7% increase.",    "SKU-1234-A, SKU-5678-B, SKU-9012-C",]print(f"{'Tokens':>7}  {'$/1M input':>11}  Content")print("-" * 70)for s in TEST_STRINGS:    tokens = len(encoding.encode(s))    cost_per_m = tokens * 3.0 / 1e6  # Sonnet input price    display = s[:55].replace("\n", " ")    if len(s) > 55:        display += "..."    print(f"{tokens:>7}  ${cost_per_m:>10.8f}  {display}")

## Section 6: Break ItThree bugs injected into slices of the same request log. For each scenario, diagnosefrom the raw data alone before reading the reveal. This mirrors how you would be handed alog excerpt in an interview and asked to figure out what is wrong.

### Break It 1: Duplicate calls (token volume doubled)Requests `req-00100` through `req-00119` have roughly 2x the normal token counts. Theretry path is not idempotent: it double-executes the call, so both input and outputtokens are doubled, but it is still logged as a single request.**Hint 1**: Look at the request log timestamps. Same input, same model, doubled tokenswithin seconds = duplicate.**Hint 2**: Compare the token counts of the affected window against the baseline average.**Production impact**: Monthly bill doubles for 4% of traffic. At scale, that is thousandsof dollars per month for zero additional value.**Interview follow-up**: "Your agent's monthly bill tripled. Walk me through how you woulddiagnose it."

#### Your diagnosis (graded)

In [ ]:
# Your diagnosis: what the data shows, what it rules out, and what you would do
# about it. Sentences, not notes -- 60 words or more.
MY_DUPLICATE_CALL_DIAGNOSIS = """Replace this with your diagnosis."""


MY_DUPLICATE_CALL_DIAGNOSIS = check("ch05-diagnose-duplicate-calls", MY_DUPLICATE_CALL_DIAGNOSIS)

#### Fix: idempotency key

In [8]:
def call_with_idempotency(request_id: str, already_processed: set, fn):
    '''The fix: check an idempotency key BEFORE executing, so a duplicate trigger (a retry,
    a double-submit) short-circuits instead of re-running the expensive call.'''
    if request_id in already_processed:
        return "skipped -- already processed this request_id"
    already_processed.add(request_id)
    return fn()


processed = set()
print("Simulating the same trigger firing twice for req-00100:")
print(" ", call_with_idempotency("req-00100", processed, lambda: "executed the real call"))
print(" ", call_with_idempotency("req-00100", processed, lambda: "executed the real call"))
print("\nOnly one real execution happened -- the second trigger was recognized and skipped")
print("before it could double the token volume.")


Simulating the same trigger firing twice for req-00100:
  executed the real call
  skipped -- already processed this request_id

Only one real execution happened -- the second trigger was recognized and skipped
before it could double the token volume.


### Break It 2: Unbounded context growthRequests `req-00200` through `req-00229` show input tokens growing linearly across thesession. The conversational agent appends every prior turn to the prompt with notruncation.**Hint 1**: Plot input_tokens across the window. Linear growth = unbounded history.**Hint 2**: Check inference_time_ms: it scales with input tokens, so it grows in lockstep.**Production impact**: A 30-turn conversation costs 30x more per call than a single-turncall. Long conversations that should cost pennies cost dollars.**Interview follow-up**: "Your agent's cost scales linearly with conversation length. Howdo you cap it without losing important context?".

#### Your diagnosis (graded)

In [ ]:
# Your diagnosis: what the data shows, what it rules out, and what you would do
# about it. Sentences, not notes -- 60 words or more.
MY_CONTEXT_GROWTH_DIAGNOSIS = """Replace this with your diagnosis."""


MY_CONTEXT_GROWTH_DIAGNOSIS = check("ch05-diagnose-context-growth", MY_CONTEXT_GROWTH_DIAGNOSIS)

#### Fix: context window policy

In [11]:
def apply_context_window_policy(history: list, max_turns: int = 6) -> list:
    '''The fix: cap how much conversation history gets re-sent, instead of appending
    forever.'''
    return history[-max_turns:]


long_history = [f"turn {i}: ..." for i in range(30)]
print(f"Full history: {len(long_history)} turns")
print(f"After the window policy: {len(apply_context_window_policy(long_history))} turns re-sent per call")
print("\nInput tokens per call now stay roughly flat regardless of how long the conversation runs,")
print("instead of growing without bound.")


Full history: 30 turns
After the window policy: 6 turns re-sent per call

Input tokens per call now stay roughly flat regardless of how long the conversation runs,
instead of growing without bound.


### Break It 3: Queueing spikeRequests `req-00300` through `req-00329`. Latency jumped from ~2s to ~12s. This is theclassic interview framing: "latency jumped, what is the first thing you investigate?"Individual requests are normal (normal token counts, normal inference time). Butqueue_depth spikes, so queue_time_ms dominates total latency.**Hint 1**: Use the four-stage breakdown. Which stage's share changed?**Hint 2**: Check input_tokens and output_tokens: if they are normal, the problem is notthe prompt or the model.**Production impact**: Every request waits behind an ever-growing line. Users see 12-secondlatency for a task that normally takes 2 seconds.**Interview follow-up**: "Latency went from 2s to 12s. You have the four-stage breakdown.What is the first thing you check?".

#### Your diagnosis (graded)

In [ ]:
# Your diagnosis: what the data shows, what it rules out, and what you would do
# about it. Sentences, not notes -- 60 words or more.
MY_QUEUEING_DIAGNOSIS = """Replace this with your diagnosis."""


MY_QUEUEING_DIAGNOSIS = check("ch05-diagnose-queueing", MY_QUEUEING_DIAGNOSIS)

#### Fix: stage share analysis

In [14]:
def summarize_stage_share(requests: list) -> dict:
    '''The diagnostic step itself: decompose a slice back into per-stage totals, the same
    way profile_latency() did for the whole log, so a spike shows up as a shift in *which*
    stage dominates rather than just a bigger total.'''
    stages = ["queue_time_ms", "network_time_ms", "inference_time_ms", "generation_time_ms"]
    totals = {stage: sum(r[stage] for r in requests) for stage in stages}
    grand_total = sum(totals.values())
    return {stage: totals[stage] / grand_total * 100 for stage in stages}


print("Stage share, normal window (requests 270-299):")
for stage, pct in summarize_stage_share(log[270:300]).items():
    print(f"  {stage:20s} {pct:5.1f}%")

print("\nStage share, spike window (requests 300-329):")
for stage, pct in summarize_stage_share(scenario_3_log[300:330]).items():
    print(f"  {stage:20s} {pct:5.1f}%")


Stage share, normal window (requests 270-299):
  queue_time_ms          5.5%
  network_time_ms        1.3%
  inference_time_ms      5.8%
  generation_time_ms    87.4%

Stage share, spike window (requests 300-329):
  queue_time_ms         76.5%
  network_time_ms        0.4%
  inference_time_ms      1.3%
  generation_time_ms    21.8%


#### Fix: backpressureThe fix for a genuine queueing spike is capacity and backpressure, not per-requestoptimization. Shed load past a depth threshold instead of letting queue time growunbounded.

In [15]:
def admit_with_backpressure(queue_depth: int, max_queue_depth: int = 15) -> str:
    '''The fix: shed load past a depth threshold instead of letting every request queue
    indefinitely behind an unbounded backlog.'''
    if queue_depth > max_queue_depth:
        return "rejected -- 503, retry with backoff (protects requests already in flight)"
    return "admitted"


print("Without backpressure, every one of these would queue and add to total_latency_ms:")
for depth in [5, 12, 25, 38]:
    print(f"  queue_depth={depth:3d}  ->  {admit_with_backpressure(depth)}")


Without backpressure, every one of these would queue and add to total_latency_ms:
  queue_depth=  5  ->  admitted
  queue_depth= 12  ->  admitted
  queue_depth= 25  ->  rejected -- 503, retry with backoff (protects requests already in flight)
  queue_depth= 38  ->  rejected -- 503, retry with backoff (protects requests already in flight)


### Fine-tuning concepts (no training, conceptual only)This subsection is deliberately conceptual. No GPU, no training run, nothing to execute.The goal is being able to speak accurately about when fine-tuning is the right lever,since in agent-engineering work the far more common answer is "better prompting, betterretrieval, or better tool design."**When fine-tuning is the right lever**: a stable, repeated behavior change that promptingcannot reliably produce. A consistent output format at high volume, a narrow domainvocabulary, a specific tool-calling style. Not for a one-off task or something still beingiterated on.**LoRA** (Hu et al., 2021): freeze original weights, train a small pair of low-rankmatrices injected alongside them. Far fewer trainable parameters, a much smaller artifactto store per task, and the base model can be shared across many LoRA adapters.**QLoRA** (Dettmers et al., 2023): LoRA + quantize the frozen base model to 4-bitprecision during training. Cuts GPU memory enough to fine-tune large models on a singleconsumer-class GPU.**RLHF** (Christiano et al., 2017; Ouyang et al., 2022): how base language models getshaped into models that follow instructions. The reward signal (a judgment about responsequality) is the same "have a model judge a response" pattern used in Chapter 3'sfaithfulness scoring and LLM-as-judge eval harnesses.

## Section 7: Interview Q&A### Question 1: "Your agent costs $500/day. How do you reduce it without hurting quality?"**Model answer**: Start with the profiler. Decompose cost into input tokens (are promptsunnecessarily long? is context growing unbounded?), output tokens (is the model verbose?are responses longer than needed?), and call volume (are there duplicate calls? retriesthat should not be retried?). Then apply levers in order of effort: prompt caching (zerocode change if the provider supports it), model routing (cheap model for simple tasks),context truncation (cap conversation history), and only then consider fine-tuning orarchitectural changes.### Question 2: "Explain the difference between inference latency and generation latency."**Model answer**: Inference latency (prefill) is the time to process the input prompt. Itscales with input length and is done in parallel. Generation latency (decode) is the timeto produce output tokens, one at a time, autoregressively. A long prompt with a shortanswer has high inference latency and low generation latency. A short prompt with a longanswer has the opposite. The fix for each is different: shorter prompts or caching forinference, shorter outputs or a faster model for generation.### Question 3: "When is prompt caching worth the implementation complexity?"**Model answer**: When you have a large prefix that repeats across many calls. The break-even depends on three numbers: prefix size (larger = more savings per hit), call volume(more calls = faster payback), and cache hit rate (depends on how often the prefix actuallyrepeats). A 2,000-token system prompt reused across 10,000 calls saves ~$54 at Sonnetpricing. A 200-token prefix reused across 50 calls saves almost nothing. The implementationcost is near-zero if the provider supports it natively.### Question 4: "When would you fine-tune vs. use a bigger model with better prompting?"**Model answer**: Fine-tune when: (1) the behavior is stable and repeated (not still beingiterated on), (2) prompting cannot reliably produce the behavior (format consistency athigh volume, domain-specific vocabulary), and (3) the training data exists and is clean.Use a bigger model with better prompting when: the task is still changing, you need thefix shipped in minutes not hours, or the behavior is prompt-expressible but your currentmodel is too weak to follow the instructions reliably. Fine-tuning is a commitment; promptingis a conversation.

### Cold-diagnosis drillAnswer each of these on your own, in writing, before checking the model answers.

1. Cold diagnosis. You are told: "our per-request GPU cost doubled overnight, no modelor traffic-volume change was deployed." Walk through what you would check, in order.2. Cold diagnosis. You are told: "p99 latency jumped from about 2 seconds to about 12seconds starting this morning." Using the four-stage breakdown, what is the first thingyou would look at?3. Judgment call. A teammate proposes fine-tuning a model to fix a chatbot that keepsgiving answers in the wrong tone. What would you ask before agreeing?4. Conceptual. Where does RLHF actually show up in an agent engineer's day-to-day work?What is the closest thing to RLHF's core mechanic elsewhere in this course?

In [20]:
from agentlib.self_check import drill as open_drill

drill = open_drill(5)
drill.questions()

Chapter 5 written drill — 4 questions

1. Cold diagnosis: per-request GPU cost doubled overnight, no model or traffic change
2. Cold diagnosis: p99 latency jumped from ~2s to ~12s
3. Judgment call: fine-tuning to fix chatbot tone
4. Conceptual: where does RLHF show up for an agent engineer, if at all?


#### Answering theseWrite your answer into the slot for each question, run the cell, then use `drill.check(n)`to see your answer and the model answer side by side. `check(n)` will not show you ananswer until you have written one of your own. If you want it anyway, `drill.reveal(n)` isthere and makes no judgement.

In [21]:
# One slot per question. Replace the placeholder text, then run this cell.
# Anything under 25 words, or left as the placeholder, is not recorded.

# Question 1
drill.attempt(1, '''
(Your answer here.)
''')

# Question 2
drill.attempt(2, '''
(Your answer here.)
''')

# Question 3
drill.attempt(3, '''
(Your answer here.)
''')

# Question 4
drill.attempt(4, '''
(Your answer here.)
''')

print()
drill.status()

  1. not recorded — it is still the placeholder
  2. not recorded — it is still the placeholder
  3. not recorded — it is still the placeholder
  4. not recorded — it is still the placeholder

Chapter 5: 0/4 answered
  still open: [1, 2, 3, 4]


In [22]:
# Your answer, then the model answer. Change the number to work through the rest.
drill.check(1)

Question 1 has no recorded answer yet.

  Cold diagnosis: per-request GPU cost doubled overnight, no model or traffic change

Write one with attempt() first. Reading the model answer before you have committed to your own turns this into a reading exercise -- once you have seen it you can no longer find out what you actually knew.
(If you really want it anyway: reveal(1).)


## Section 8: References1. Hu, E. J., et al. (2021). "LoRA: Low-Rank Adaptation of Large Language Models."   https://arxiv.org/abs/2106.096852. Dettmers, T., et al. (2023). "QLoRA: Efficient Finetuning of Quantized Language   Models." https://arxiv.org/abs/2305.143143. Christiano, P., et al. (2017). "Deep Reinforcement Learning from Human Preferences."   https://arxiv.org/abs/1706.037414. Ouyang, L., et al. (2022). "Training language models to follow instructions with human   feedback." (InstructGPT) https://arxiv.org/abs/2203.021555. Anthropic pricing: https://docs.anthropic.com/en/docs/about-claude/pricing6. OpenAI tiktoken: https://github.com/openai/tiktokenRelated chapters:- Chapter 4 (caching, retry cost)- Chapter 8 (design decisions around cost and model selection)- Chapter 9 (deployment cost monitoring)

## Next: Chapter 6, Security and SafeguardsThis chapter was about the cost of not failing. Chapter 6 is about what happens when anagent's input is adversarial: prompt injection, defense in depth, and least-privilege toolscoping.